[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/llms/09-langgraph.ipynb)

# LangGraph — Agentes con estado y flujos condicionales

StateGraph, nodos, aristas condicionales, persistencia con MemorySaver
y human-in-the-loop para agentes de producción.

In [ ]:
# pip install langgraph langchain-anthropic
import os
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
import operator

llm = ChatAnthropic(model='claude-haiku-4-5-20251001', max_tokens=512)
print('LangGraph + Claude Haiku listos')

## 1. Grafo simple — clasificador con enrutamiento

In [ ]:
class EstadoSoporte(TypedDict):
    mensaje: str
    categoria: str
    respuesta: str
    escalado: bool

def clasificar(estado: EstadoSoporte) -> EstadoSoporte:
    """Nodo 1: clasifica el tipo de consulta."""
    resp = llm.invoke([
        SystemMessage(content='Clasifica la consulta. Responde SOLO con una palabra: facturacion, tecnico, cancelacion, otro'),
        HumanMessage(content=estado['mensaje'])
    ])
    return {**estado, 'categoria': resp.content.strip().lower()}

def resolver_facturacion(estado: EstadoSoporte) -> EstadoSoporte:
    resp = llm.invoke([
        SystemMessage(content='Eres especialista en facturación. Responde brevemente.'),
        HumanMessage(content=estado['mensaje'])
    ])
    return {**estado, 'respuesta': resp.content, 'escalado': False}

def resolver_tecnico(estado: EstadoSoporte) -> EstadoSoporte:
    resp = llm.invoke([
        SystemMessage(content='Eres soporte técnico nivel 1. Responde con pasos claros.'),
        HumanMessage(content=estado['mensaje'])
    ])
    return {**estado, 'respuesta': resp.content, 'escalado': False}

def escalar(estado: EstadoSoporte) -> EstadoSoporte:
    return {
        **estado,
        'respuesta': f'Tu consulta sobre "{estado["mensaje"][:50]}..." ha sido escalada a un agente humano.',
        'escalado': True
    }

def enrutar(estado: EstadoSoporte) -> Literal['facturacion', 'tecnico', 'escalar']:
    cat = estado['categoria']
    if 'facturaci' in cat:
        return 'facturacion'
    elif 'tecnic' in cat:
        return 'tecnico'
    return 'escalar'

# Construir el grafo
grafo = StateGraph(EstadoSoporte)
grafo.add_node('clasificar', clasificar)
grafo.add_node('facturacion', resolver_facturacion)
grafo.add_node('tecnico', resolver_tecnico)
grafo.add_node('escalar', escalar)

grafo.set_entry_point('clasificar')
grafo.add_conditional_edges('clasificar', enrutar)
grafo.add_edge('facturacion', END)
grafo.add_edge('tecnico', END)
grafo.add_edge('escalar', END)

app = grafo.compile()

# Probar con diferentes consultas
consultas = [
    '¿Cuándo se renueva mi suscripción y cuánto me cobrarán?',
    'La aplicación se cierra sola al abrir un PDF.',
    'Quiero reclamar por discriminación en el servicio.',
]

for consulta in consultas:
    resultado = app.invoke({'mensaje': consulta, 'categoria': '', 'respuesta': '', 'escalado': False})
    escalado_str = '🔴 ESCALADO' if resultado['escalado'] else '✅ Resuelto'
    print(f'[{resultado["categoria"].upper()}] {escalado_str}')
    print(f'Q: {consulta}')
    print(f'A: {resultado["respuesta"][:150]}...')
    print()

## 2. Agente con historial — usando MemorySaver

In [ ]:
from langchain_core.messages import BaseMessage

class EstadoChat(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]  # reducer: concatena mensajes

def chat_con_claude(estado: EstadoChat) -> EstadoChat:
    system = SystemMessage(content='Eres un asistente de análisis legal. Mantén coherencia con la conversación.')
    respuesta = llm.invoke([system] + estado['messages'])
    return {'messages': [respuesta]}

grafo_chat = StateGraph(EstadoChat)
grafo_chat.add_node('chat', chat_con_claude)
grafo_chat.set_entry_point('chat')
grafo_chat.add_edge('chat', END)

# MemorySaver persiste el estado entre invocaciones del mismo thread_id
memory = MemorySaver()
app_chat = grafo_chat.compile(checkpointer=memory)

config = {'configurable': {'thread_id': 'sesion-001'}}

def enviar(mensaje: str) -> str:
    resultado = app_chat.invoke(
        {'messages': [HumanMessage(content=mensaje)]},
        config=config
    )
    return resultado['messages'][-1].content

# Conversación con memoria persistente
print('Turn 1:', enviar('El contrato de Acme Corp vence el 1 de agosto de 2026.'))
print('\nTurn 2:', enviar('¿Cuándo debería enviar el aviso de 90 días?'))
print('\nTurn 3:', enviar('¿Y si quiero negociar una prórroga de 6 meses?'))

# Ver historial completo almacenado
estado_guardado = app_chat.get_state(config)
print(f'\nHistorial en memoria: {len(estado_guardado.values["messages"])} mensajes')

## 3. Human-in-the-loop con interrupt_before

In [ ]:
from langgraph.types import interrupt

class EstadoAprobacion(TypedDict):
    solicitud: str
    analisis: str
    aprobado: bool | None
    accion_ejecutada: str

def analizar_solicitud(estado: EstadoAprobacion) -> EstadoAprobacion:
    resp = llm.invoke([
        SystemMessage(content='Analiza esta solicitud legal. Indica riesgo (alto/medio/bajo) y recomendación.'),
        HumanMessage(content=estado['solicitud'])
    ])
    return {**estado, 'analisis': resp.content}

def esperar_aprobacion_humana(estado: EstadoAprobacion) -> EstadoAprobacion:
    """Este nodo pausa la ejecución y espera aprobación humana."""
    print(f'\n⏸️  PAUSA — Se requiere aprobación humana')
    print(f'Solicitud: {estado["solicitud"]}')
    print(f'Análisis: {estado["analisis"][:200]}...')

    # interrupt() pausa el grafo y devuelve el control al caller
    aprobado = interrupt({'pregunta': '¿Aprobar esta acción? (True/False)', 'analisis': estado['analisis']})
    return {**estado, 'aprobado': aprobado}

def ejecutar_accion(estado: EstadoAprobacion) -> EstadoAprobacion:
    if estado['aprobado']:
        accion = f'✅ Acción ejecutada: procesada la solicitud "{estado["solicitud"][:50]}"'
    else:
        accion = f'❌ Acción rechazada por el usuario'
    return {**estado, 'accion_ejecutada': accion}

grafo_hitl = StateGraph(EstadoAprobacion)
grafo_hitl.add_node('analizar', analizar_solicitud)
grafo_hitl.add_node('aprobar', esperar_aprobacion_humana)
grafo_hitl.add_node('ejecutar', ejecutar_accion)
grafo_hitl.set_entry_point('analizar')
grafo_hitl.add_edge('analizar', 'aprobar')
grafo_hitl.add_edge('aprobar', 'ejecutar')
grafo_hitl.add_edge('ejecutar', END)

memory_hitl = MemorySaver()
app_hitl = grafo_hitl.compile(checkpointer=memory_hitl)

config_hitl = {'configurable': {'thread_id': 'hitl-001'}}

estado_inicial = {
    'solicitud': 'Rescisión anticipada del contrato con Acme Corp con penalización de 50.000€',
    'analisis': '', 'aprobado': None, 'accion_ejecutada': ''
}

# Primera ejecución — se pausa en 'aprobar'
try:
    resultado = app_hitl.invoke(estado_inicial, config=config_hitl)
except Exception as e:
    print(f'Grafo pausado (esperando aprobación): {type(e).__name__}')

# Simular aprobación humana y reanudar
print('\n▶️  Reanudando con aprobación = True')
app_hitl.update_state(config_hitl, {'aprobado': True}, as_node='aprobar')
resultado_final = app_hitl.invoke(None, config=config_hitl)
print(resultado_final.get('accion_ejecutada', 'Sin acción'))

## 4. Streaming de tokens y eventos

In [ ]:
# Streaming con stream_mode='values' (estado en cada paso)
print('Streaming con mode=values:')
config_stream = {'configurable': {'thread_id': 'stream-001'}}

for evento in app_chat.stream(
    {'messages': [HumanMessage(content='Resume en 2 frases qué es LangGraph')]},
    config=config_stream,
    stream_mode='values',
):
    ultimo_msg = evento['messages'][-1]
    if hasattr(ultimo_msg, 'content') and ultimo_msg.content:
        print(f'  [{type(ultimo_msg).__name__}]: {ultimo_msg.content[:80]}...')

# Streaming de tokens en tiempo real
print('\nStreaming de tokens en tiempo real:')
for chunk, metadata in app_chat.stream(
    {'messages': [HumanMessage(content='Di "Hola" en 5 idiomas')]},
    config={'configurable': {'thread_id': 'stream-002'}},
    stream_mode='messages',
):
    if hasattr(chunk, 'content') and chunk.content:
        print(chunk.content, end='', flush=True)
print()